# Bölüm 2 — Önerme Mantığı

Matematiksel önerme (proposition), doğru ya da yanlış olabilen bir ifadedir.
Bu bölüm `pytop.logic` modülünün sağladığı `Proposition`, bağlaçlar ve
sonlu niceleyicileri tanıtır; ardından ayrılma aksiyomlarını bu araçlarla
nasıl ifade edebileceğimizi gösterir.

In [1]:
from pytop import (
    Proposition,
    negate, conjunction, disjunction, implies, iff,
    for_all, there_exists, unique_exists,
)

## 1. Önerme ve Doğruluk Değeri

`Proposition(name, truth_value)` isimli bir önerme nesnesi oluşturur.
`bool(p)` ve `p.truth_value` doğruluk değerini verir.

In [2]:
p = Proposition("p", True)
q = Proposition("q", False)
r = Proposition("r", True)

print(f"p : {p.truth_value}")
print(f"q : {q.truth_value}")
print(f"bool(p): {bool(p)}")

p : True
q : False
bool(p): True


## 2. Mantıksal Bağlaçlar

| Fonksiyon | Sembol | Anlam |
|-----------|--------|-------|
| `negate(p)` | ¬p | p değil |
| `conjunction(p, q)` | p ∧ q | p ve q |
| `disjunction(p, q)` | p ∨ q | p veya q |
| `implies(p, q)` | p → q | p ise q |
| `iff(p, q)` | p ↔ q | p ancak ve ancak q |

In [3]:
print("neg(p)     :", negate(p).truth_value)
print("p and q    :", conjunction(p, q).truth_value)
print("p or q     :", disjunction(p, q).truth_value)
print("p -> q     :", implies(p, q).truth_value)
print("p <-> q    :", iff(p, q).truth_value)
print("p <-> p    :", iff(p, p).truth_value)
print("p and q and r:", conjunction(p, q, r).truth_value)

neg(p)     : False
p and q    : False
p or q     : True
p -> q     : False
p <-> q    : False
p <-> p    : True
p and q and r: False


`implies(p, q)` yalnızca $p=\text{True},\ q=\text{False}$ durumunda yanlıştır;
diğer üç durumda boş doğruluk (vacuous truth) nedeniyle doğrudur.

## 3. Doğruluk Tablosu

In [4]:
print(f"{'p':>5}  {'q':>5}  {'neg_p':>5}  {'p&q':>5}  {'p|q':>5}  {'p->q':>6}  {'p<->q':>6}")
print("-" * 52)
for tv_p in (True, False):
    for tv_q in (True, False):
        pp = Proposition("p", tv_p)
        qq = Proposition("q", tv_q)
        row = (
            pp.truth_value, qq.truth_value,
            negate(pp).truth_value,
            conjunction(pp, qq).truth_value,
            disjunction(pp, qq).truth_value,
            implies(pp, qq).truth_value,
            iff(pp, qq).truth_value,
        )
        print("  ".join(f"{str(v):>5}" for v in row))

    p      q  neg_p    p&q    p|q    p->q   p<->q
----------------------------------------------------
 True   True  False   True   True   True   True
 True  False  False  False   True  False  False
False   True   True  False   True   True  False
False  False   True  False  False   True   True


## 4. Önemli Tautolojiler

**Teorem (De Morgan):**
$\neg(p \wedge q) \leftrightarrow \neg p \vee \neg q$ ve
$\neg(p \vee q) \leftrightarrow \neg p \wedge \neg q$

**Teorem (Karşıt):**
$(p \to q) \leftrightarrow (\neg q \to \neg p)$

In [5]:
pairs = [(True, True), (True, False), (False, True), (False, False)]

def check_tautology(name, func):
    ok = all(func(Proposition("p", tp), Proposition("q", tq)).truth_value
             for tp, tq in pairs)
    print(f"{name}: {'tautoloji' if ok else 'DEGIL'}")

check_tautology(
    "neg(p&q) <-> neg_p|neg_q",
    lambda p, q: iff(negate(conjunction(p, q)), disjunction(negate(p), negate(q)))
)
check_tautology(
    "neg(p|q) <-> neg_p&neg_q",
    lambda p, q: iff(negate(disjunction(p, q)), conjunction(negate(p), negate(q)))
)
check_tautology(
    "(p->q) <-> (neg_q->neg_p)",
    lambda p, q: iff(implies(p, q), implies(negate(q), negate(p)))
)

def check_single(name, func):
    ok = all(func(Proposition("p", tv)).truth_value for tv in (True, False))
    print(f"{name}: {'tautoloji' if ok else 'DEGIL'}")

check_single("p -> p", lambda p: implies(p, p))
check_single("p | neg_p (excluded middle)", lambda p: disjunction(p, negate(p)))

neg(p&q) <-> neg_p|neg_q: tautoloji
neg(p|q) <-> neg_p&neg_q: tautoloji
(p->q) <-> (neg_q->neg_p): tautoloji
p -> p: tautoloji
p | neg_p (excluded middle): tautoloji


## 5. Niceleyiciler

```python
for_all(carrier, predicate)       # ∀x ∈ carrier : P(x)
there_exists(carrier, predicate)  # ∃x ∈ carrier : P(x)
unique_exists(carrier, predicate) # ∃!x ∈ carrier : P(x)
```

In [6]:
X = [1, 2, 3, 4, 5]

print("for_all(X, x>0)       :", for_all(X, lambda x: x > 0))
print("for_all(X, x>2)       :", for_all(X, lambda x: x > 2))
print("there_exists(X, x>4)  :", there_exists(X, lambda x: x > 4))
print("there_exists(X, x>5)  :", there_exists(X, lambda x: x > 5))
print("unique_exists(X, x==3):", unique_exists(X, lambda x: x == 3))
print("unique_exists(X, x>3) :", unique_exists(X, lambda x: x > 3))
print("unique_exists(X, x>5) :", unique_exists(X, lambda x: x > 5))

for_all(X, x>0)       : True
for_all(X, x>2)       : False
there_exists(X, x>4)  : True
there_exists(X, x>5)  : False
unique_exists(X, x==3): True
unique_exists(X, x>3) : False
unique_exists(X, x>5) : False


`unique_exists` yalnızca tam bir eleman sayıldığında `True` döner;
sıfır veya birden fazlada `False`.

## 6. Topoloji Uygulaması — Ayrılma Aksiyomları

**T0 (Kolmogorov):**
$\forall x \neq y \in X,\ \exists$ açık $U : (x \in U,\ y \notin U)$ veya $(y \in U,\ x \notin U)$

**T1 (Fréchet):**
$\forall x \neq y \in X,\ \exists$ açık $U : x \in U,\ y \notin U$

In [7]:
from pytop import sierpinski_space, discrete_topology, indiscrete_topology

def is_t0_logic(space):
    pts   = list(space.carrier)
    opens = list(space.topology)
    return for_all(
        [(x, y) for x in pts for y in pts if x != y],
        lambda pair: there_exists(
            opens, lambda U: (pair[0] in U) != (pair[1] in U)
        )
    )

def is_t1_logic(space):
    pts   = list(space.carrier)
    opens = list(space.topology)
    return for_all(
        [(x, y) for x in pts for y in pts if x != y],
        lambda pair: there_exists(
            opens, lambda U: pair[0] in U and pair[1] not in U
        )
    )

spaces = {
    "Sierpinski": sierpinski_space(),
    "Discrete  ": discrete_topology(0, 1),
    "Indiscrete": indiscrete_topology(0, 1),
}

print(f"{'Uzay':<12}  T0     T1")
print("-" * 26)
for name, sp in spaces.items():
    print(f"{name}  {str(is_t0_logic(sp)):<5}  {str(is_t1_logic(sp)):<5}")

Uzay          T0     T1
--------------------------
Sierpinski  True   False
Discrete    True   True 
Indiscrete  False  False


`for_all`/`there_exists` ile yazılan T0/T1 tanımları, `pytop`'un
`is_t0` ve `is_t1` yüklemlerinin sonuçlarıyla örtüşür.

## Alıştırmalar

1. Beş önerme değeriyle tam bir doğruluk tablosu oluşturun; `implies` kaç
   `(p, q)` çiftinde `False` döner?

2. `unique_exists([0,1,2,3,4], predicate)` değerini `True` yapan üç farklı
   koşul yazın.

3. T2 (Hausdorff) aksiyomunu `for_all` ve `there_exists` kullanarak kodlayın:
   $\forall x \neq y,\ \exists$ açık $U,V : x \in U,\ y \in V,\ U \cap V = \emptyset$.
   Sierpiński, discrete ve indiscrete için test edin.

4. *(Teori)* `implies(p, q)` neden yalnızca $p=T,\ q=F$ durumunda yanlıştır?
   Boş doğruluk (vacuous truth) kavramını açıklayın.

5. *(Teori)* De Morgan yasalarını ispatlayın:
   $\neg(p \wedge q) \leftrightarrow \neg p \vee \neg q$.